<a href="https://colab.research.google.com/github/CSRobba/WearEver_DH25/blob/main/scrape_reddit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overview

### Goal:
Scrape posts from Reddit that ask questions about AI's environmental Impact. Specifically, we are looking for

* Posts made after November 30th, 2022
* Posts made in English
* Posts relating to AI's environmental footprint

### Date: January 27th, 2026

### Notes:
Task Breakdown:
* 1) Identify all the posts that are relevant to us: [PushPull.io](https://www.pullpush.io/) should help us identify potentially relevant posts.

* 2) Download the data (the post, and its associated comments) for all of these posts

# Done
* Identify the ideal time interval: [N] It appears that three days is a good interval.
* Save progress as we go along: [N] Saving each post result as a json file.
* Figured out how the logger works. [N]
* Implemented search for multiple different versions of post titles. For instance, in addition to searching for ```"AI"``` and ```"water"``` and ```"?```, also search for ```"AI"``` and ```"energy."``` [N]
* Implemented a tracker for queries we need to run, queries that have been run, and only starting where we left off. [N]

# TO-DOs

* Use git with this code. [C]
* For dates that contain more than 100 posts, re-run the code at smaller time intervals. [C]

# Configurations

## Install Packages

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=9c52b9d0d30174cb1d29aec3bd4c5269f652d6a3e5b0364ee71c8eff6ebc7aa9
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


## Import Packages

In [ ]:
import requests
from datetime import datetime, timedelta
from google.colab import drive
import itertools
import json
from langdetect import detect, DetectorFactory
import logging
import pandas
import os
import time
from tqdm import tqdm
import sys

## Set Default Values

In [ ]:
# Mount Drive
drive.mount('/content/drive')
# TODO: Change this based on Google Drive
WORK_DIR = '/content/drive/My Drive/Research Projects/01 First Author Projects/05 AI Environmental Footprint/04 Online Data Scraping'
os.chdir(WORK_DIR)

Mounted at /content/drive


In [ ]:
# 1. Create the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 2. Create the format
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s')

# 3. File Handler
log_file = os.path.join(WORK_DIR, "scraper.log")
file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(formatter)

# 4. Stream Console Handler
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)

# 5. Add handlers to the logger
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

In [ ]:
DetectorFactory.seed = 0
TIME_INTERVAL = 7 #Three days

In [ ]:
BASE_URL = "https://api.pullpush.io/reddit/search/submission/"
COMMENT_ID_URL = "https://api.pullpush.io/reddit/submission/comment_ids/"

# Helper Functions

In [ ]:
def _generate_queries_all():

  ai_terms = ["AI",
              "artificial intelligence",
              #"generative AI", #duplicate with "AI"
              #"GenAI", #duplicate with "AI"
              "LLM",
              "Large Language Model",
              "ChatGPT",
              "DeepSeek",
              "Gemini",
              "Llama",
              "Claude",
              "data center"]
  env_terms = ["water use",
               "water usage",
               "water footprint",
               "environment footprint",
               "sustainability",
               "environment impact",
               "environment effect",
               "environment cost",
               "environment harm",
               "environment negative",
               "pollution",
               #"e-waste", #duplicate with "waste"
               #"ewaste", #duplicate with "waste"
               "waste",
               #"electronic waste", #duplicate with "waste"
               "energy consumption",
               "energy use",
               "energy usage",
               "energy footprint",
               "carbon emission"]

  start = datetime(2022, 11, 30)
  end = datetime(2026, 2, 2) #datetime.now().date()
  start_dates = pandas.date_range(start, end, freq=f"{TIME_INTERVAL}D")

  df_queries = pandas.DataFrame(list(itertools.product(ai_terms, env_terms, start_dates)), columns = ["ai", "env", "start"])
  df_queries["end"] = df_queries["start"] + timedelta(TIME_INTERVAL)

  return df_queries

In [ ]:
def _generate_queries_already_run():

  if os.path.isfile(os.path.join(WORK_DIR, "queries_run.csv")):
    queries_already_run = pandas.read_csv(os.path.join(WORK_DIR, "queries_run.csv"))
    queries_already_run = queries_already_run.drop("Unnamed: 0", axis = 1)
    queries_already_run["start"] = pandas.to_datetime(queries_already_run["start"])
    queries_already_run["end"] = pandas.to_datetime(queries_already_run["end"])
    return queries_already_run

  return pandas.DataFrame([], columns = ["ai", "env", "start", "end", "count"])

In [ ]:
def generate_queries_to_run():

  queries_all = _generate_queries_all()
  queries_past = _generate_queries_already_run()

  queries_to_run = queries_all.merge(queries_past,
                                     how = "left",
                                     on = ["ai", "env", "start", "end"])
  queries_to_run = queries_to_run.infer_objects(copy=False).fillna(-1)

  return queries_to_run

In [ ]:
def format_isodate_to_date(dt):

  dt_obj = datetime.fromisoformat(dt).date()
  df_string = dt_obj.strftime('%Y_%m_%d')
  return df_string

In [ ]:
def save_submission(result):

  file_path = os.path.join(WORK_DIR, f"raw/{result['id']}_{result['query_term'].replace(' ', '_').replace('-', '_').lower()}_{format_isodate_to_date(result['created'])}.json")

  # Use 'w' mode for writing
  with open(file_path, 'w', encoding='utf-8') as f:
      json.dump(result, f, indent=4)

In [ ]:
# Ensure language detection is consistent
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

In [ ]:
def get_pushpull_comment_ids(post_id):
  # Fetch Comment IDs for this post
  try:
    c_resp = requests.get(f"{COMMENT_ID_URL}{post_id}")
    comment_ids = c_resp.json().get('data', [])
  except:
      comment_ids = []

  return comment_ids

In [ ]:
def process_pushpull_submission(data, params, get_comments = False):

  results = []

  for post in data:
    title = post.get('title', '')

    if is_english(title):
      post_id = post.get('id')


      result = {
        'query_term': params['title'],
        'title': title,
        'id': post_id,
        'body': post.get('selftext'),
        'url': post.get('url'),
        'url_overridden_by_dest': post.get('url_overridden_by_dest'),
        'created': datetime.fromtimestamp(post.get('created_utc')).isoformat(),
        'author': post.get('author'),
        'subreddit': post.get('subreddit'),
        'num_comments': post.get('num_comments'),
        'ups': post.get('ups'),
        'upvote_ratio': post.get('upvote_ratio'),
        'view_count': post.get('view_count')
      }

      if get_comments:
        comment_ids = get_pushpull_comment_ids(post_id)
        result['comment_ids'] = comment_ids

      save_submission(result)
      results.append(result)

  return results

In [ ]:
def get_pushpull_submission(params, row):
  try:
      response = requests.get(BASE_URL, params=params)
      response.raise_for_status()
      data = response.json().get('data', [])

      logger.info(f"Processing Term: {params['title']} For {row['start'].date()}: Found {len(data)} potential matches.")

      return process_pushpull_submission(data, params)

  except Exception as e:
      logger.info(f"Error on {params['title']} For {row['start'].date()}: {e}")

      raise ValueError()

In [ ]:
def run_query(row):
  after = int(row["start"].timestamp())
  before = int(row["end"].timestamp())

  logger.info(f"Starting scrape from {row['start'].date()} to {(row['end']).date()}...")

  # TO BE ADJUSTED
  params = {
        'title': f"{row['ai']} {row['env']}", # include this tag if we want to only search in the title of the post
        #'both': f"{row['ai']} {row['env']}", # include this tag if we want to only search in both title and body of the post
        'after': after,
        'before': before,
        'size': 100,
        'sort': 'asc'
  }

  try:
    results = get_pushpull_submission(params, row)
    row["count"] = len(results)
  except ValueError as e:
    pass

  return row

In [ ]:
def get_reddit_data(queries):

  for indx, row in tqdm(queries.iterrows(), total=queries.shape[0]):
    if row["count"] == -1:

      row = run_query(row)
      queries.at[indx, "count"] = row["count"]
      queries.to_csv(os.path.join(WORK_DIR, "queries_run.csv"))
      time.sleep(4)

  return queries

# Main

In [ ]:
queries_to_run = generate_queries_to_run()

In [ ]:
get_reddit_data(queries_to_run)

 70%|███████   | 19756/28220 [00:01<00:00, 12144.33it/s]

2026-02-18 16:02:30,278 [INFO] Starting scrape from 2024-06-26 to 2024-07-03...


INFO:__main__:Starting scrape from 2024-06-26 to 2024-07-03...


2026-02-18 16:02:30,913 [INFO] Processing Term: Llama sustainability For 2024-06-26: Found 0 potential matches.


INFO:__main__:Processing Term: Llama sustainability For 2024-06-26: Found 0 potential matches.


...

INFO:__main__:Starting scrape from 2024-10-02 to 2024-10-09...


2026-02-19 00:01:25,002 [INFO] Processing Term: data center environment harm For 2024-10-02: Found 0 potential matches.


INFO:__main__:Processing Term: data center environment harm For 2024-10-02: Found 0 potential matches.
 95%|█████████▌| 26823/28220 [7:59:00<1:47:50,  4.63s/it]

2026-02-19 00:01:29,275 [INFO] Starting scrape from 2024-10-09 to 2024-10-16...


INFO:__main__:Starting scrape from 2024-10-09 to 2024-10-16...
